In [1]:
"""
Week 1 Reference Solution: MNIST Classification with PyTorch
Diffusion Models from Scratch — SoC 2026

Run on Google Colab: Runtime > Change runtime type > T4 GPU
"""

# ============================================================
# SECTION 0: Imports and reproducibility
# ============================================================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# SECTION 1: Device setup
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ============================================================
# SECTION 2: Custom Dataset class (wraps torchvision MNIST)
# ============================================================
class MNISTWrapper(Dataset):
    def __init__(self, root="./data", train=True, download=True):
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])
        self.mnist = datasets.MNIST(
            root=root, train=train, download=download, transform=self.transform,
        )

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        image, label = self.mnist[idx]
        return image, label


# ============================================================
# SECTION 3: Data splits and DataLoaders
# ============================================================
BATCH_SIZE = 128
VAL_FRACTION = 0.1

full_train = MNISTWrapper(train=True, download=True)
test_set   = MNISTWrapper(train=False, download=True)

val_size   = int(len(full_train) * VAL_FRACTION)
train_size = len(full_train) - val_size
train_set, val_set = random_split(
    full_train, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)
print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)


# ============================================================
# SECTION 4: Model definition (simple MLP)
# ============================================================
class MLP(nn.Module):
    def __init__(self, in_dim=784, hidden=256, out_dim=10, p_drop=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


model = MLP().to(device)
print(model)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")


# ============================================================
# SECTION 5: Loss and optimizer
# ============================================================
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


# ============================================================
# SECTION 6: Evaluation helper
# ============================================================
@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = loss_fn(logits, y)
        total_loss    += loss.item() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum().item()
        total_samples += x.size(0)
    return total_loss / total_samples, total_correct / total_samples


# ============================================================
# SECTION 7: Training loop with logging and best-checkpoint saving
# ============================================================
NUM_EPOCHS = 10
CKPT_PATH  = "best_model.pt"
best_val_acc = 0.0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running_loss, running_correct, running_samples = 0.0, 0, 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

        running_loss    += loss.item() * x.size(0)
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_samples += x.size(0)

    train_loss = running_loss / running_samples
    train_acc  = running_correct / running_samples
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "epoch": epoch,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "val_acc": val_acc,
        }, CKPT_PATH)
        print(f"  -> new best val_acc; checkpoint saved to {CKPT_PATH}")


# ============================================================
# SECTION 8: Reload best checkpoint and evaluate on test set
# ============================================================
ckpt = torch.load(CKPT_PATH, map_location=device)
fresh_model = MLP().to(device)
fresh_model.load_state_dict(ckpt["model_state"])

test_loss, test_acc = evaluate(fresh_model, test_loader, loss_fn, device)
print(f"\nLoaded checkpoint from epoch {ckpt['epoch']} (val_acc={ckpt['val_acc']:.4f})")
print(f"FINAL TEST ACCURACY: {test_acc:.4f}")
assert test_acc >= 0.97, f"Test accuracy {test_acc:.4f} below required 0.97"
print("Deliverable threshold met.")

Using device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 20.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 507kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.71MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.7MB/s]


Train: 54000 | Val: 6000 | Test: 10000
MLP(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=256, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.2, inplace=False)
    (7): Linear(in_features=256, out_features=10, bias=True)
  )
)
Total parameters: 269,322
Epoch 01/10 | train_loss=0.3056 train_acc=0.9091 | val_loss=0.1436 val_acc=0.9562
  -> new best val_acc; checkpoint saved to best_model.pt
Epoch 02/10 | train_loss=0.1311 train_acc=0.9600 | val_loss=0.1051 val_acc=0.9670
  -> new best val_acc; checkpoint saved to best_model.pt
Epoch 03/10 | train_loss=0.0964 train_acc=0.9699 | val_loss=0.1047 val_acc=0.9680
  -> new best val_acc; checkpoint saved to best_model.pt
Epoch 04/10 | train_loss=0.0805 train_acc=0.9748 | val_loss=0.0797 val_acc=0.9728
  -> new best val_acc; checkpoint saved to best_model.pt
Epoch 05/1